# Lesson 13 - Rock Paper Scissors Vision

In this lesson, SpiderPi uses MediaPipe hand recognition to play rock, paper, scissors with you.

## What We Will Do

1. Get ready
2. Count down with the arm
3. Return to `camera.center_all()`
4. Capture and read the hand sign
5. Compare moves
6. Show the winner


In [ ]:
from lesson_loader import setup
setup()
from student_robot_v2 import bot
import random

bot.help()


In [ ]:
# Get ready
bot.display.text("Ready", seconds=1.0)
bot.lights.blue()
bot.sound.beep(2200, 0.08)
bot.buzzer.off()
bot.camera.center_all()
bot.arm.open()


In [ ]:
# SpiderPi picks a move and counts down
robot_move = random.choice(["rock", "paper", "scissors"])
print("Robot move:", robot_move)

def show_robot_move(move):
    if move == "rock":
        bot.arm.close()
    elif move == "paper":
        bot.arm.open()
    else:
        bot.arm.half_open()

def countdown_and_show(move):
    beats = [
        ("Ready", 24, 12, 1100),
        ("Rock", 22, 12, 1300),
        ("Paper", 20, 12, 1500),
        ("Scissors", 18, 12, 1700),
        ("Go", 14, 10, 2100),
    ]
    bot.camera.center_all()
    bot.arm.open()
    for word, up_height, down_height, freq in beats:
        bot.display.text(word, seconds=0.03)
        bot.sound.beep(freq, 0.03)
        bot.arm.move(0, 15, up_height, seconds=0.03)
        bot.arm.move(0, 15, down_height, seconds=0.03)
    bot.buzzer.off()
    bot.camera.center_all()

countdown_and_show(robot_move)


In [ ]:
# Capture the hand sign
hand_result = bot.vision.recognize_hands(show=True)
print(hand_result)

possible_moves = hand_result.get("game_moves", [])
player_move = possible_moves[0] if possible_moves else None
print("Player move:", player_move)


In [ ]:
# Compare the moves
if player_move is None:
    bot.lights.yellow()
    bot.display.text("Try Again", seconds=1.0)
else:
    show_robot_move(robot_move)
    bot.display.rps(robot_move, player_move, seconds=3.0)

def winner(player, robot):
    if player is None:
        return "retry"
    if player == robot:
        return "draw"
    wins = {
        ("rock", "scissors"),
        ("paper", "rock"),
        ("scissors", "paper"),
    }
    return "player" if (player, robot) in wins else "robot"

result = winner(player_move, robot_move)
print("Result:", result)


In [ ]:
# Show the winner
if result == "retry":
    bot.lights.yellow()
    bot.display.text("Try Again", seconds=1.0)
    bot.sound.beep(900, 0.12)
    bot.sound.beep(700, 0.16)
elif result == "draw":
    bot.lights.purple()
    bot.display.text("Draw", seconds=1.0)
    bot.body.twist()
    bot.sound.beep(1200, 0.10)
    bot.sound.beep(1200, 0.10)
elif result == "player":
    bot.lights.red()
    bot.display.text("You Win", seconds=1.0)
    bot.body.backward(0.4)
    bot.sound.beep(1400, 0.08)
    bot.sound.beep(1800, 0.08)
    bot.sound.beep(2200, 0.14)
else:
    bot.lights.green()
    bot.display.text("I Win", seconds=1.0)
    bot.body.dance()
    bot.sound.beep(2200, 0.08)
    bot.sound.beep(1800, 0.08)
    bot.sound.beep(1400, 0.14)

bot.buzzer.off()
bot.arm.home()
bot.body.stop()


## Challenge

Try one or more of these upgrades:

- change the countdown arm heights or speed
- make SpiderPi choose a special move sound for each result
- keep score across three rounds
- make different gestures trigger different action groups